<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c26-header.png" alt="Nextia Learning · AI System Design and Cost Engineering" width="100%">

# Cost and capacity plan from real demand data

**[Cost and capacity plan from real demand data](https://learning.nextia-ai.com/courses/system-design/m07/cost-and-capacity-plan-from-real-demand-data/)** · AI System Design and Cost Engineering · Module 7, case study 1 of 2 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** turn a real public demand series into a cost and capacity plan. You check the data and measure the daily pattern, the busiest minute and the growth. You compare three ways to describe the peaks on a year that you kept aside. Then you feed the result to the course's design pack as `demand.toml`. The pack gives the cost range, the first limit that the design reaches, and how far away each limit is.

| | |
|---|---|
| **Time** | About 2 hours with the case-study page. The notebook runs in about 1 minute. |
| **Needs** | An internet connection for the setup cells. No account, no key, no money. The first cell installs pandas 3.0.6 and matplotlib. |
| **You should know** | The whole course, most of all [Demand assumptions](https://learning.nextia-ai.com/courses/system-design/m01/demand-assumptions/), [Workload arithmetic](https://learning.nextia-ai.com/courses/system-design/m03/workload-arithmetic/), [Unit economics](https://learning.nextia-ai.com/courses/system-design/m03/unit-economics/) and [Growth plan](https://learning.nextia-ai.com/courses/system-design/m05/growth-plan/). |
| **Data** | Boston's 311 service requests from residents, January 2022 to July 2025, counted per hour (Analyze Boston, ODC Public Domain Dedication and License 1.0). The course's design pack at the end of Module 6, unchanged. |
| **Tested** | Python 3.12.13 with pandas 3.0.6, numpy 2.5.3 and matplotlib 3.11.2, in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/system-design/m07/cost-and-capacity-plan-from-real-demand-data/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C26/M07-L01-cost-and-capacity-plan/cost-and-capacity-plan-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6, the version that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see another version, restart the kernel (in Colab: **Runtime › Restart session**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 matplotlib
import pandas; print("pandas", pandas.__version__)

## Setup: get the data and the design pack

Run the next cell. It downloads 13 files from the Nextia labs repository into a folder `cost-plan`, and checks each file's checksum (SHA-256):

- `boston-311-hourly.csv` (about 0.9 MB): the residents' 311 requests in every hour from January 2022 to July 2025;
- `design-pack/`: the calculator of the course's design pack (`costmodel/`) and its input files `design.toml`, `prices.toml` and `measured.toml`, from the snapshot `end-of-m06`. The notebook does not change them. It writes only a new `demand.toml`.

You should see `Ready: 13 files`. If a copy of a file is next to this notebook, the cell uses that copy.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `cost-plan` (in Colab: the **Files** panel on the left), then run the cell again.

Data: "311 Service Requests", City of Boston, Analyze Boston (data.boston.gov/dataset/311-service-requests), ODC Public Domain Dedication and License 1.0. Nextia Learning counted the requests per hour and froze a copy; the dataset card says how.

In [ ]:
# Setup: download the files into cost-plan/ and check them. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/"
FILES = {
    "boston-311-hourly.csv":
        ("C26/M07-L01-cost-and-capacity-plan/boston-311-hourly.csv", "2ad790797890f35a67f4a9edd0b5c057e695edd30a0dd2b9f8ba51488f8238d6"),
    "design-pack/costmodel/__init__.py":
        ("C26/snapshots/end-of-m06/costmodel/__init__.py", "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"),
    "design-pack/costmodel/__main__.py":
        ("C26/snapshots/end-of-m06/costmodel/__main__.py", "366e2878e9475294ea6e8f10887d24ae33f1540c37eac7185e73b32bc09a78b1"),
    "design-pack/costmodel/inputs.py":
        ("C26/snapshots/end-of-m06/costmodel/inputs.py", "068290268ef35691edc8175dbb7eb43a9112696c5fafebad079a0d2f4857f0ce"),
    "design-pack/costmodel/demand.py":
        ("C26/snapshots/end-of-m06/costmodel/demand.py", "fc8ff2fcebbf46fa2841308da5b647b435f2a05cafc97b998c1928900459c266"),
    "design-pack/costmodel/design.py":
        ("C26/snapshots/end-of-m06/costmodel/design.py", "aca54dffb9e7d2a0928cc07368be6453f2d6379c12ecf2d89e99bc3a517265aa"),
    "design-pack/costmodel/workload.py":
        ("C26/snapshots/end-of-m06/costmodel/workload.py", "698ae088db2d626e0a7036ec32cc6bbc4ce2f3c2ee6e9ae3b5a56362ca15358d"),
    "design-pack/costmodel/costs.py":
        ("C26/snapshots/end-of-m06/costmodel/costs.py", "0873cd7017e7510ee2131779c1d4f2b18340b85e95726e9007282ac0851eef3e"),
    "design-pack/costmodel/units.py":
        ("C26/snapshots/end-of-m06/costmodel/units.py", "91bde8a0faa0cea081ac7a4d8ddd8f30b338f5af833510f683e727c81a8194de"),
    "design-pack/costmodel/growth.py":
        ("C26/snapshots/end-of-m06/costmodel/growth.py", "a9dfe6450f19b12321f7bfbed9ed2940f8aac64f49e02303a72e6afd282855e1"),
    "design-pack/design.toml":
        ("C26/snapshots/end-of-m06/design.toml", "b8a2ab337d03fdbfb0d632fadd80b995415eca3ddc0bf3c5c8af819115e9ca03"),
    "design-pack/prices.toml":
        ("C26/snapshots/end-of-m06/prices.toml", "616f56b3dbc71b826b75e7e2a6706920a57391503d565fc67d80baa1fa6f3727"),
    "design-pack/measured.toml":
        ("C26/snapshots/end-of-m06/measured.toml", "8a756d9c2cef1e1260a8378103085098f2e1b3ee5bee633a13103d113c6b93fb"),
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("cost-plan").resolve()
(WORK / "design-pack" / "costmodel").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for name, (path, expected) in FILES.items():
    target = WORK / name
    if not (target.exists() and sha256(target) == expected):
        if (HERE / name).exists():
            shutil.copy(HERE / name, target)
        else:
            url = LABS + path
            try:
                urllib.request.urlretrieve(url, target)
            except OSError:  # Python without certificates (python.org on macOS): try curl
                if subprocess.run(["curl", "-fsSL", "-o", str(target), url]).returncode:
                    raise SystemExit(f"Could not download {url}. Check your internet connection.")
    if sha256(target) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder cost-plan and run again.")
os.chdir(WORK)
print(f"Ready: {len(FILES)} files")

## Setup: helpers

The next cell loads pandas and matplotlib, and defines the case study's functions. They are the same functions that the case-study page shows:

- `load_hours()` and `month_table()` read the file and make one row per month;
- `shape_ranges()`, `volume_range()` and `growth_range()` turn months into the ranges of `demand.toml`;
- `busiest_minute()` is the pack's arithmetic from a day to its busiest minute;
- `demand_toml()` writes the file, and `pack()` runs the calculator as on the command line;
- `limits()` and `headroom()` read the pack's three growth signals, and `plan()` runs the whole method.

`expect(what, yours, expected)` compares your answer with the expected one and says "too high" or "too low", without showing the answer. Run the cell. It prints nothing.

In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

QUOTA_ALERT, STORAGE_ALERT = 0.7, 0.8  # the pack's lines (design.toml [growth])


def load_hours(path):
    """The hourly sample: one row per hour, local time."""
    hours = pd.read_csv(path, parse_dates=["hour"])
    hours["month"] = hours.hour.dt.to_period("M")
    return hours


def month_table(hours):
    """One row per month: the volume and the peaks that the pack needs."""
    m = hours.groupby("month").agg(requests=("requests", "sum"), busiest_hour=("requests", "max"),
                                   busiest_minute=("busiest_minute", "max"))
    m["days"] = m.index.days_in_month
    m["mean_day"] = m.requests / m.days
    m["peak_hour_share"] = m.busiest_hour / m.mean_day              # busiest hour / an average day
    m["peak_minute_factor"] = m.busiest_minute * 60 / m.busiest_hour  # busiest minute / that hour's average
    return m


def three(low, base, high, digits=3):
    return [round(float(low), digits), round(float(base), digits), round(float(high), digits)]


def shape_ranges(months):
    """The measured shape: the quietest, the typical (median) and the busiest month."""
    return {k: three(months[k].min(), months[k].median(), months[k].max())
            for k in ("peak_hour_share", "peak_minute_factor")}


def volume_range(months):
    """Requests per day: the quietest month, the average of the period, the busiest month."""
    return three(months.mean_day.min(), months.requests.sum() / months.days.sum(), months.mean_day.max(), 0)


def growth_range(months):
    """Monthly growth. base: the last 12 months against the 12 before. low and high: the month that
    changed least and most against the same month a year before. The pack takes no negative growth."""
    year_on_year = (months.requests / months.requests.shift(12)).dropna().to_numpy()
    per_month = year_on_year ** (1 / 12) - 1
    total = (months.requests.iloc[-12:].sum() / months.requests.iloc[-24:-12].sum()) ** (1 / 12) - 1
    return {"measured": three(per_month.min(), total, per_month.max(), 4),
            "pack": three(max(0, per_month.min()), max(0, total), max(0, per_month.max()), 4)}


def busiest_minute(mean_day, share, factor):
    """The pack's arithmetic: an average day x the busiest hour's share x the minute factor / 60."""
    return mean_day * share * factor / 60


SHAPES = {  # three ways to describe the peaks (the volume is the same for all three)
    "average": {"peak_hour_share": [round(1 / 24, 4)] * 3, "peak_minute_factor": [1.0] * 3},
    "borrowed": {"peak_hour_share": [0.10, 0.123, 0.20], "peak_minute_factor": [3.0, 5.35, 8.0]},  # the course's Larkfield shape
}

CITY = {  # the city's assistant: assumed values, each with its reason
    "staff_users": ([0, 0, 0], "assumed: this plan covers residents only; staff questions are out of scope"),
    "staff_questions_per_day": ([0, 0, 0], "assumed: no staff users in this plan"),
    "customer_questions_per_day": ([1.0, 1.5, 3.0], "assumed: a question and a follow-up, as the course assumes for "
                                   "Larkfield's customers; 3 if residents also ask before they file"),
    "documents": ([150, 300, 600], "assumed: the city's service pages and rules that the assistant answers from"),
    "words_per_document": ([300, 600, 1200], "assumed: a service page is longer than a help-desk policy"),
    "document_changes_per_month": ([10, 30, 60], "assumed: seasonal pages (snow, street cleaning) change often"),
}


def demand_toml(volume, shape, growth, period, shape_note):
    """The text of demand.toml for one tenant, the city's residents. Every value has its evidence."""
    def r(values):
        return "[" + ", ".join(f"{v:g}" for v in values) + "]"
    lines = [
        "# Demand for a city's public question-answering assistant, from the city's 311 requests.",
        f"# Measured on {period}. Every value is a range [low, base, high] with its evidence.",
        "", "[settings]", "days_per_month = 30",
        f"peak_hour_share = {r(shape['peak_hour_share'])}",
        f"peak_minute_factor = {r(shape['peak_minute_factor'])}",
        "", "[settings.evidence]",
        f'peak_hour_share = "{shape_note} (busiest hour / an average day of its month)"',
        f'peak_minute_factor = "{shape_note} (busiest minute / its hour\'s average minute)"',
        "", "[tenants.residents]", 'name = "Residents"', "joins_in_month = 0",
        f"customers_per_day = {r(volume)}",
        f"user_growth_per_month = {r(growth)}",
    ]
    lines += [f"{k} = {r(v)}" for k, (v, _) in CITY.items()]
    lines += ["", "[tenants.residents.evidence]",
              f'customers_per_day = "verified: residents\' requests per day, {period}: the quietest month, '
              f'the average, the busiest month (one request = one resident who asks)"',
              f'user_growth_per_month = "verified: the last 12 months against the 12 before, {period}; '
              f'negative values set to 0"']
    lines += [f'{k} = "{why}"' for k, (_, why) in CITY.items()]
    return "\n".join(lines) + "\n"


def pack(*args, folder="design-pack"):
    """Run the design pack's calculator, as on the command line: python -m costmodel <args>."""
    out = subprocess.run([sys.executable, "-m", "costmodel", *args], cwd=folder, capture_output=True, text=True)
    return (out.stdout + out.stderr).rstrip()


def limits(scenario="base", month=0, usage=1.0, folder="design-pack"):
    """The pack's three growth signals for one scenario, month and usage multiplier, as shares of their lines."""
    folder = str(Path(folder).resolve())
    if folder not in sys.path:
        sys.path.insert(0, folder)
    from costmodel import costs, demand, design, inputs, workload  # noqa: F401  (the pack, unchanged)
    d, g = demand.load_demand(Path(folder) / "demand.toml"), design.load_design(Path(folder) / "design.toml")
    m, p = inputs.load_measured(Path(folder) / "measured.toml"), inputs.load_prices(Path(folder) / "prices.toml")
    budget = inputs.read_toml(Path(folder) / "design.toml")["growth"]["budget_usd_per_month"]
    c, w = costs.monthly(d, g, m, p, scenario, month, usage)
    per_answer = m["answer_small_tokens_in"].value + m["answer_small_tokens_out"].value
    used = sum(w.storage_gb.values()) - w.storage_gb["files"] + w.storage_growth_gb_per_month * month
    return {"questions_per_day": w.questions_per_month / d.days_per_month,
            "calls": w.model_calls_peak_minute / m["provider_quota_requests_per_minute"].value,
            "tokens": w.model_calls_peak_minute * per_answer / m["provider_quota_tokens_per_minute"].value,
            "storage": used / g.database_storage_gb, "cost": sum(c.values()), "budget": budget}


def headroom(scenario="base", folder="design-pack", top=200.0):
    """How many times today's questions before each line: quota 70%, storage 80%, the budget."""
    def crosses(name, x):
        s = limits(scenario, 0, x, folder)
        return {"quota": max(s["calls"], s["tokens"]) / QUOTA_ALERT, "storage": s["storage"] / STORAGE_ALERT,
                "budget": s["cost"] / s["budget"]}[name] >= 1
    out = {}
    for name in ("quota", "storage", "budget"):
        lo, hi = 0.01, top
        if not crosses(name, hi):
            out[name] = None   # not within `top` times today's questions
            continue
        for _ in range(60):     # bisection: the smallest multiplier that crosses the line
            mid = (lo + hi) / 2
            lo, hi = (lo, mid) if crosses(name, mid) else (mid, hi)
        out[name] = round(hi, 2)
    return out


def plan(hours, as_of, folder="design-pack"):
    """The whole method on the data before `as_of` (a month, "2025-08"): checks, demand.toml, the pack,
    the first limit. The volume comes from the last 12 months; the shape and the growth from all months."""
    months = month_table(hours[hours.hour < pd.Timestamp(as_of)])
    if len(months) < 24:
        raise ValueError(f"{len(months)} months: the growth needs at least 24")
    if (months.days * 24 > months.index.to_series().map(lambda p: (hours.month == p).sum())).any():
        raise ValueError("a month has missing hours: check the file")
    last12 = months.iloc[-12:]
    first, last = months.index[0].strftime("%B %Y"), months.index[-1].strftime("%B %Y")
    model = {"volume": volume_range(last12), "growth": growth_range(months)["pack"], "shape": shape_ranges(months)}
    Path(folder, "demand.toml").write_text(demand_toml(
        model["volume"], model["shape"], model["growth"], f"{first} to {last}",
        "verified: measured on the city's 311 requests"))
    result = {"model": model, "scenarios": {}}
    for scenario in ("low", "base", "high"):
        rows = [limits(scenario, month, folder=folder) for month in range(13)]
        first_signal = next(((mo, name) for mo, r in enumerate(rows) for name, hit in (
            ("provider quota", max(r["calls"], r["tokens"]) >= QUOTA_ALERT),
            ("database storage", r["storage"] >= STORAGE_ALERT),
            ("budget", r["cost"] >= r["budget"])) if hit), None)
        result["scenarios"][scenario] = {"cost_month_0": round(rows[0]["cost"], 2), "cost_month_12": round(rows[12]["cost"], 2),
                                         "quota_month_12": round(max(rows[12]["calls"], rows[12]["tokens"]), 3),
                                         "first_signal": first_signal, "headroom": headroom(scenario, folder)}
    print(f"Plan from {first} to {last}: residents' requests per day {model['volume']}, "
          f"growth per month {model['growth']}")
    for scenario, r in result["scenarios"].items():
        signal = f"{r['first_signal'][1]} in month {r['first_signal'][0]}" if r["first_signal"] else "none in 12 months"
        print(f"  {scenario:<4}  US${r['cost_month_0']:>7.2f} now, US${r['cost_month_12']:>7.2f} in month 12; "
              f"quota {r['quota_month_12']:.0%} in month 12; first signal: {signal}; "
              f"quota line at {r['headroom']['quota']}x today's questions")
    return result


# Helpers for the notebook: charts, and a check that hides the answer.
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

def expect(what, yours, expected, tolerance=0.005):
    """Compare one number with the expected one (within 0.5%)."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif abs(float(yours) - expected) <= tolerance * max(1, abs(expected)):
        print(f"Check passed: {what} is {float(yours):g}.")
    else:
        print(f"Not yet: {what} is {float(yours):g}, which is too {'high' if float(yours) > expected else 'low'}.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A city's 311 line answers residents: bulky-item pickup, a missed collection, a broken street light, the parking rules on a snow day. Rosa leads the city's digital services. She wants a **public question-answering assistant** on the 311 website and app. It answers from the city's own service pages, with citations, like the course's document assistant. Before she asks for money, she asks Jamal, an architect, three questions:

- "What will it cost per month, as a range?"
- "What limit do we hit first, and when?"
- "When must we change the design, and how will we know?"

The assistant does not exist yet, so nobody has its traffic. But the city's 311 requests are public. Jamal uses them as the demand: **one resident who files a request today is one resident who asks the assistant**. The design is the course's design at the end of Module 6, unchanged. The [case-study page](https://learning.nextia-ai.com/courses/system-design/m07/cost-and-capacity-plan-from-real-demand-data/) tells the full story; this notebook runs every step.

## 2. The data

Load the file, and look at its size, its first rows and the time range.

In [ ]:
hours = load_hours("boston-311-hourly.csv")
print(hours.shape)
print(hours.head(3).to_string(index=False))
print(hours.hour.min(), "to", hours.hour.max())

> **Check.** You should see `(31392, 7)`: one row per hour for 1,308 days, plus the `month` column that `load_hours()` adds. The file runs from `2022-01-01 00:00:00` to `2025-07-31 23:00:00`.

One row is one hour, in Boston local time. The columns:

| Column | What it means |
|---|---|
| `requests` | requests from residents (app, phone call, web form) in that hour, repeated rows removed |
| `requests_raw` | the same, before removing the repeated rows |
| `busiest_minute` | the most requests from residents in one minute of that hour (repeats removed) |
| `busiest_minute_raw` | the same, before removing the repeats |
| `staff_requests` | requests that city staff created themselves |

## 3. First checks

Each check answers one question that the plan depends on: who asks, are there repeats, are hours missing, and when do people ask.

In [ ]:
years = hours.groupby(hours.hour.dt.year)[["requests", "requests_raw", "staff_requests"]].sum()
print(years)
print("hours without a request:", (hours.requests == 0).sum())
print("repeated rows, share of the volume:", f"{1 - hours.requests.sum() / hours.requests_raw.sum():.2%}")
print("busiest minute with repeats:", hours.busiest_minute_raw.max(), "without:", hours.busiest_minute.max())
of_day = hours.hour.dt.hour
print("requests between 8:00 and 20:00:", f"{hours.requests[(of_day >= 8) & (of_day < 20)].sum() / hours.requests.sum():.1%}")

> **Check.** You should see about 131,700 requests in 2022, 163,800 in 2023, 145,400 in 2024 and 70,500 in the seven months of 2025. Repeats are only 0.16% of the volume, but they raise the busiest minute from 26 to 51. About 83% of the requests arrive between 8:00 and 20:00.

**What this tells us.**

- **Staff create about half of all 311 requests** (`staff_requests`). Staff would not use a residents' assistant, so the sample counts residents only.
- **Repeats hardly change the volume, but they double the busiest minute.** The city's app sometimes sends one report several times. The cost depends on the volume; the provider quota depends on the busiest minute. So the plan uses the counts without repeats.
- **About 2,900 night hours have no request.** They are real zeros, and the file has a row for each.
- **83% of the requests arrive between 8:00 and 20:00.** The pack's `design.toml` assumes 12 busy hours a day (base). The data agrees.
- The file stops at **July 2025**. From August 2025 the city moved requests to a new system, with other columns. A mix of the two would look like a fall in demand.

## 4. Split the time

A plan made from the past must work in the future. So Jamal uses three periods, in time order:

- **design months**, 2022 and 2023: to choose the method;
- **check year**, 2024: to test the choice **once**;
- **new months**, January to July 2025: to run the finished method end to end.

In [ ]:
months = month_table(hours)
design = months.loc["2022-01":"2023-12"]
check = months.loc["2024-01":"2024-12"]
new = months.loc["2025-01":"2025-07"]
print(len(design), "design months,", len(check), "check months,", len(new), "new months")
print(months.loc["2023-01":"2023-04"].round(3).to_string())

> **Check.** You should see `24 design months, 12 check months, 7 new months`, and four rows of the month table. In March 2023, an average day had 383.5 requests, the busiest hour had 62 and the busiest minute had 6.

## 5. The daily pattern and the months

Two pictures decide the plan: when in the day people ask, and how the volume changes from month to month. Run the cell.

In [ ]:
weekday = hours.hour.dt.dayofweek < 5
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.6))
for mask, label in [(weekday, "weekdays"), (~weekday, "weekends")]:
    a.plot(hours[mask].groupby(hours.hour.dt.hour[mask]).requests.mean(), label=label)
a.set(xlabel="hour of the day", ylabel="requests per hour (average)", title="The day"); a.legend()
b.plot(months.index.to_timestamp(), months.mean_day)
b.axvspan(pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31"), alpha=0.15, label="check year")
b.set(xlabel="month", ylabel="requests per day (average)", title="The months"); b.legend()
plt.tight_layout(); plt.show()

> **Check.** On weekdays, requests climb from under 1 an hour at 3:00 to about 39 an hour at 9:00, and stay above 30 until 16:00. Weekends peak later and lower, at about 22 an hour. The months rise from about 303 a day in January 2022 to 566 in August 2023, then fall. The check year averages 397, and January 2025 has 274.

**What this tells us.** The day has a strong peak, so an average minute describes no real minute. And the volume does not grow in a straight line: it rose by about a quarter in one year, then fell for two. Growth is the least certain number of the plan.

## 6. The idea on one month

The pack goes from a day to its busiest minute in two steps ([Demand assumptions](https://learning.nextia-ai.com/courses/system-design/m01/demand-assumptions/)). An average day × the **busiest hour's share** gives the busiest hour. The busiest hour ÷ 60 × the **busiest minute factor** gives the busiest minute. The pack multiplies an **average** day. So Jamal measures both numbers against an average day of the same month.

Do it by hand for March 2023: an average day of 383.5 requests, a busiest hour of 62, and a busiest minute of 6.

> **Predict.** Before you run the cell, compute the share (62 ÷ 383.5) and the factor (6 × 60 ÷ 62). Then multiply back: does the pack's arithmetic give 6 again?

In [ ]:
march = months.loc["2023-03"]
share = march.busiest_hour / march.mean_day
factor = march.busiest_minute * 60 / march.busiest_hour
print(f"share {share:.3f}, factor {factor:.2f}, busiest minute {busiest_minute(march.mean_day, share, factor):.2f}")

> **Check.** You should see `share 0.162, factor 5.81, busiest minute 6.00`. The two numbers describe the month's peak exactly, so the pack can rebuild it.

## 7. Three ways to describe the peaks

Every month gives one share and one factor. A plan needs a range: the quietest month (low), the typical month (base, the median) and the busiest month (high). Jamal compares three ways to fill the two lines of `demand.toml`:

- **average**: no peak at all, every minute is an average minute (share 1/24, factor 1). This is the baseline: the arithmetic of a spreadsheet.
- **borrowed**: the shape that the course uses for Larkfield, from the scaling course's help-desk tickets (also measured against an average day).
- **measured**: the ranges measured on the 24 design months.

In [ ]:
measured = shape_ranges(design)
shapes = {"average": SHAPES["average"], "borrowed": SHAPES["borrowed"], "measured": measured}
for name, s in shapes.items():
    print(f"{name:9} share {s['peak_hour_share']}, factor {s['peak_minute_factor']}")

> **Check.** The measured share is `[0.148, 0.173, 0.403]` and the factor `[4.337, 5.424, 9.496]`. The high values come from August 2023 and August 2022, months with sudden runs of reports.

### The fair comparison, once

One measure decides: for each month of the check year, take that month's own average day, and predict its busiest minute with each shape. A good base case covers most months without a large margin. The check year was not used to choose anything.

> **Predict.** Which shape's base case covers more of the 12 check months?

In [ ]:
rows = []
for name, s in shapes.items():
    base = busiest_minute(check.mean_day, s["peak_hour_share"][1], s["peak_minute_factor"][1])
    high = busiest_minute(check.mean_day, s["peak_hour_share"][2], s["peak_minute_factor"][2])
    rows.append({"shape": name, "base covers": int((base >= check.busiest_minute).sum()),
                 "base (median)": round(base.median(), 2), "high covers": int((high >= check.busiest_minute).sum()),
                 "high (median)": round(high.median(), 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("observed busiest minutes, 2024:", check.busiest_minute.tolist())

> **Check.** The average covers 0 of 12 months (0.29 a minute). The borrowed shape covers 0 in its base case (4.55) and 12 in its high case (11.07). The measured shape covers 9 in its base case (6.49) and 12 in its high case (26.47). The observed busiest minutes are 5 or 6.

**What this tells us.**

- **The average describes no real minute.** It predicts 0.29 requests in the busiest minute; every month had 5 or 6.
- **The borrowed shape is too low in its base case.** It is measured correctly, against an average day, but on another workload. The help desk's busiest hour holds 0.123 of an average day; here the measured base share is 0.173. Residents' requests crowd into fewer hours than help-desk tickets. A shape belongs to its workload.
- **The measured shape fits.** Its base case is about one request above the observed values and covers 9 of 12 months. Its high case is about four times the observed peak: it is a stress case, built from the runs of reports.

### Volume and growth on the check year

The same check for the volume: the design months give requests per day (low, base, high) and a monthly growth range. Did 2024 stay inside it?

In [ ]:
volume, growth = volume_range(design), growth_range(design)
print("requests per day:", volume, " growth per month:", growth)
plan_2024 = pd.DataFrame({"low": [volume[0] * (1 + growth["pack"][0]) ** k for k in range(1, 13)],
                          "base": [volume[1] * (1 + growth["pack"][1]) ** k for k in range(1, 13)],
                          "high": [volume[2] * (1 + growth["pack"][2]) ** k for k in range(1, 13)]},
                         index=check.index)
plan_2024["actual"] = check.mean_day
inside = ((plan_2024.low <= plan_2024.actual) & (plan_2024.actual <= plan_2024.high)).sum()
print(plan_2024.round(0).astype(int).to_string())
print("months inside low..high:", inside, "  base average", round(plan_2024.base.mean()), "actual", round(check.requests.sum() / check.days.sum()))

> **Check.** Requests per day `[293.0, 405.0, 566.0]`, growth `[0.0, 0.0184, 0.0278]` a month for the pack. 11 of the 12 months are inside the range. The base case expected about 457 a day in 2024; the city had 397.

**What this tells us.** The range held, but the base case was 15% too high: the growth of 2023 (+24%) did not happen again. A growth rate from two years is a guess. So the plan does not trust it; it watches **signals** instead ([Growth plan](https://learning.nextia-ai.com/courses/system-design/m05/growth-plan/)).

## 8. The recommendation

Jamal uses the **measured shape**, with the volume of the last 12 months and the growth range of the whole history. He writes the growth plan as signals with lines, not as a forecast.

## 9. The plan on new input, end to end

`plan(hours, as_of)` runs the whole method on the data before `as_of`. It checks that no hour is missing, measures the ranges and writes `demand.toml`. Then it runs the pack for the three scenarios over 12 months. It finds the first signal and the **headroom**: how many times today's questions the design takes before each line.

First, the plan that Jamal could make at the start of 2025, before the new months:

In [ ]:
before = plan(hours, "2025-01")

> **Check.** You should see a plan from January 2022 to December 2024, with requests per day `[281.0, 397.0, 457.0]`. The high case reaches the quota's line in month 2.

Now with the seven new months. This run leaves its `demand.toml` in the pack:

In [ ]:
latest = plan(hours, "2025-08")
print(Path("design-pack/demand.toml").read_text())

> **Check.** Requests per day `[274.0, 347.0, 440.0]`, growth `[0.0, 0.0, 0.0278]`. Base: US$47.45 a month, no signal in 12 months, quota line at about 9 times today's questions. High: US$59.72 now, US$61.66 in month 12; the provider quota's line in month 3.

**What this tells us.** The new months lowered the volume (397 → 347 requests a day). The base cost fell by only US$0.35, because almost all of it is fixed. The high case's quota signal moved from month 2 to month 3.

### The pack's own output

`demand.toml` is now the city's. Run the pack's commands as on the command line ([Set up the design pack](https://learning.nextia-ai.com/courses/system-design/m01/set-up-the-design-pack/)):

In [ ]:
for command in ["demand", "costs", "units", "scenarios"]:
    print(f"$ python -m costmodel {command}")
    print(pack(*command.split()))
    print()

> **Check.** `demand` prints 274 / 520 / 1,320 questions a day. `costs` prints a total of US$47.45, with the API replica (US$27.43) and the database server (US$13.50) first. `units` says 95% fixed and US$0.003039 per question, everything included. `scenarios`: US$39.92 / 47.45 / 59.72 now, US$39.92 / 47.45 / 61.66 in month 12.

In [ ]:
print(pack("growth", "--scenario", "high"))

> **Check.** The high case's token share rises from 65% in month 0 to 90% in month 12. The last line says `First signal: provider quota in month 3`. Storage stays at 1% or less, and the cost far below the budget of US$150.

### The headroom

> **Predict.** In the base case, how many times today's questions can the design take before the provider quota's line (70%)? Before the budget? Before storage?

In [ ]:
for scenario in ["base", "high"]:
    print(scenario, headroom(scenario))

> **Check.** Base: quota about 9.0 times, budget about 43 times, storage `None` (not within 200 times). High: quota about 1.1 times, budget about 18 times.

**What this tells us.** In the base case, the first limit is the provider quota, and it is far away: about nine times today's questions. So the question that matters is not "how many residents file a request", but "how many residents will ask the assistant **without** filing one". That is unknown until launch. It becomes the plan's main signal.

### The growth plan

| Signal | Line | Step | Source |
|---|---|---|---|
| tokens in the busiest minute | 70% of the quota | ask the provider for a higher quota; queue residents' questions at the peak | the pack's `growth` |
| questions a day (the assistant's log) | 9 × today's base, about 4,700 | rerun this plan with the real log | the headroom |
| the city's 311 requests a day | above the plan's high (440) for a month | rerun `plan()` | the 311 data |
| monthly cost | the budget, US$150 | the biggest driver first: the API replica | the pack's `growth` |

The [case-study page](https://learning.nextia-ai.com/courses/system-design/m07/cost-and-capacity-plan-from-real-demand-data/) explains each step and links the lessons.

## 10. Change one thing: keep the repeats

What if Jamal had kept the repeated rows? Rebuild the shape with the raw busiest minutes, and compare the high case.

In [ ]:
raw_hours = hours.assign(busiest_minute=hours.busiest_minute_raw)
raw_shape = shape_ranges(month_table(raw_hours).loc["2022-01":"2025-07"])
clean_shape = shape_ranges(months)
for name, s in [("without repeats", clean_shape), ("with repeats", raw_shape)]:
    print(f"{name:16} factor {s['peak_minute_factor']}, high busiest minute for 440 residents x 3 questions:",
          round(busiest_minute(440 * 3, s["peak_hour_share"][2], s["peak_minute_factor"][2]), 1))

> **Check.** With repeats, the high factor rises from 9.5 to 48.3. The high busiest minute rises from about 84 to about 428 questions: more than four times the provider quota of 100 a minute.

**What this tells us.** 0.16% of the rows would have moved the high case over the provider's limit. Clean the data that sets the peak; the volume barely notices.

## 11. A failure case: a falling demand

Demand fell in the new months. What if you write the measured growth, a negative number, into `demand.toml`?

In [ ]:
text = Path("design-pack/demand.toml").read_text()
measured_growth = growth_range(months)["measured"]
print("measured growth per month:", measured_growth)
Path("design-pack/demand.toml").write_text(text.replace("user_growth_per_month = [0, 0, 0.0278]",
                                                        f"user_growth_per_month = {measured_growth}"))
print(pack("demand"))
Path("design-pack/demand.toml").write_text(text)  # put the plan's file back

> **Check.** The pack refuses: `Input problem: 'residents.user_growth_per_month' is negative`. The last line of the cell puts the plan's file back.

**What this tells us.** The pack takes no negative ranges. That is a design choice. A falling demand lowers the cost, so the plan sets the low and base growth to 0, and the evidence says so. If you need a shrinking plan, write a smaller volume instead.

## 12. Your turn

### Task 1: one month's share

> **Your turn.** Compute the busiest hour's share of **July 2023** from `months`, as in section 6, and put it in `share_july_2023`.

In [ ]:
share_july_2023 = ...  # your code here

In [ ]:
expect("the busiest hour's share of July 2023", share_july_2023, 0.1795)

### Task 2: the quota line in questions

> **Your turn.** In the base case, how many questions a day reach the provider quota's line? Multiply the base questions a day (`limits("base")["questions_per_day"]`) by the base headroom for the quota. Put it in `questions_at_line`.

In [ ]:
questions_at_line = ...  # your code here

In [ ]:
expect("the base questions a day at the quota line", questions_at_line, 4674)

### Task 3: residents who ask more

> **Your turn.** Rosa thinks residents will ask 3 questions each, not 1.5. Change the base of `customer_questions_per_day` to 3 in `design-pack/demand.toml` (the line is `customer_questions_per_day = [1, 1.5, 3]`). Then put the base case's monthly cost, `limits("base")["cost"]`, in `cost_three_questions`. Put the file back at the end, as section 11 does.

In [ ]:
cost_three_questions = ...  # your code here

In [ ]:
expect("the base monthly cost with 3 questions per resident", cost_three_questions, 49.91)

> **Check.** If a check says "too high" or "too low", read the task again. The [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C26/M07-L01-cost-and-capacity-plan/cost-and-capacity-plan-solution.ipynb) has the answers.

## Recap

- One request from a resident stood for one resident who asks. Staff requests and repeated rows were left out: the repeats hardly change the volume, but they double the busiest minute.
- The pack multiplies an **average** day, so the busiest hour's share and the minute factor were measured against an average day of each month.
- On the check year, used once, the measured shape covered 9 of 12 months in its base case. The average and the borrowed shape covered none. The growth of the design months was 15% too high.
- The plan: US$39.92 to 59.72 a month, 95% fixed in the base case. The first limit is the provider quota: month 3 in the high case, about 9 times today's questions in the base case.
- The growth plan watches signals with lines. Its main unknown is how many residents ask without filing a request.

The knowledge checks are on the [case-study page](https://learning.nextia-ai.com/courses/system-design/m07/cost-and-capacity-plan-from-real-demand-data/).

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Design review of an open document-question service](https://learning.nextia-ai.com/courses/system-design/m07/design-review-of-an-open-document-question-service/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/system-design/m07/cost-and-capacity-plan-from-real-demand-data/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The 311 data is real public data (ODC PDDL 1.0); this sample has only counts per hour, no names or addresses.